# Preprocessing Pipeline Demo

**Purpose**: This notebook implements every stage of a pretraining data pipeline from Chapter 4 on a small synthetic corpus: Unicode normalisation, language identification, quality and repetition filters, perplexity ranking, exact and MinHash deduplication, personal-data redaction, benchmark decontamination, and a complete pipeline with per-stage drop accounting. Each section ends in assertions, so a cell that runs is a cell whose claims hold.

**Requirements**: NumPy and Matplotlib only. Everything runs on a CPU in seconds.

## 1. Setup and a Synthetic Corpus

In [1]:
import hashlib
import math
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

# A small corpus with the kinds of documents a crawl contains. Each entry is
# (id, text, language). "en"/"de" labels are the truth the identifier is tested on.
ARTICLE = (
    "The town council voted on Monday to approve the new library building after months "
    "of public consultation about its design, location and projected cost. Supporters "
    "argued that the existing branch had long been too small for the number of residents "
    "who use it, while opponents questioned whether the money would be better spent on "
    "road repairs. The vote passed by a margin of seven to four, and construction is "
    "expected to begin in the spring of next year, with the building opening in two years."
)
CORPUS = [
    ("article", ARTICLE, "en"),
    ("syndicated", ARTICLE, "en"),
    ("retitled", "Library approved. " + ARTICLE + " Read more on our website.", "en"),
    ("mojibake", ARTICLE.replace("council", "cafÃ©​ council").replace("e", "é", 3), "en"),
    (
        "german",
        "Der Stadtrat hat am Montag den Bau der neuen Bibliothek genehmigt, nachdem monatelang "
        "über Entwurf, Standort und Kosten diskutiert worden war. Befürworter argumentierten, "
        "dass die bestehende Zweigstelle für die Zahl der Nutzer seit langem zu klein sei, "
        "während Gegner fragten, ob das Geld nicht besser in die Reparatur der Straßen "
        "investiert werden sollte. Der Beschluss fiel mit sieben zu vier Stimmen, und die "
        "Bauarbeiten sollen im Frühjahr des kommenden Jahres beginnen.",
        "de",
    ),
    ("menu", "\n".join(f"- {item}" for item in ["Home", "About", "Products", "Contact", "Login"] * 12), "en"),
    ("spam", "buy cheap watches now " * 30, "en"),
    ("boilerplate", "The Town Library. Library of the Town. The library and the town. " * 15, "en"),
    ("catalogue", " ".join(f"SKU-{i:05d} {i * 7 % 100}.99" for i in range(80)), "en"),
    (
        "contact_list",
        "\n".join(f"Member {i}: member{i}@example.org, +1 (555) 010-{i:04d}" for i in range(10)),
        "en",
    ),
    (
        "prose_with_pii",
        "For questions about the schedule, email the coordinator at coordinator@example.org "
        "or call (555) 123-4567 during office hours. The server at 10.0.0.12 hosts the shared "
        "calendar, which is updated every Friday afternoon by the administrative team. Please "
        "do not forward the login details to anyone outside the department, as they are "
        "rotated monthly and the old ones stop working immediately after each rotation.",
        "en",
    ),
    (
        "leaked_benchmark",
        "Here is a question from the quiz we ran last week: What is the capital of the country "
        "whose largest city is Istanbul and whose currency is the lira? The answer, of course, "
        "is Ankara, which surprised several of the participants who had guessed Istanbul. We "
        "will run another round next month with questions on geography and chemistry, and the "
        "results will be posted on the notice board as usual by the end of the week.",
        "en",
    ),
    (
        "paraphrased_benchmark",
        "A geography teaser from last week: the lira is the money and Istanbul the biggest city, "
        "so which city is the capital? Most people guess wrong. The answer is Ankara, and it has "
        "been since the republic was founded, although Istanbul remains the commercial centre "
        "and the city most visitors see first. We will post more teasers of this kind next month "
        "for anyone who enjoyed this one and wants another challenge.",
        "en",
    ),
    (
        "harbour",
        "The harbour was quiet at dawn. A single trawler moved out past the breakwater, its "
        "lights still on. By seven the market had opened and the first crates of fish were being "
        "carried up from the quay. Nobody mentioned the storm forecast for the afternoon, though "
        "everyone had read it, and the older skippers had already decided not to go out again "
        "until it had passed. The cafe on the corner did a brisk trade in coffee all morning.",
        "en",
    ),
]
print(f"{len(CORPUS)} documents")
for doc_id, text, lang in CORPUS:
    print(f"  {doc_id:<22} {lang}  {len(text.split()):>4} words")

14 documents
  article                en    85 words
  syndicated             en    85 words
  retitled               en    92 words
  mojibake               en    86 words
  german                 de    71 words
  menu                   en   120 words
  spam                   en   120 words
  boilerplate            en   180 words
  catalogue              en   160 words
  contact_list           en    60 words
  prose_with_pii         en    62 words
  leaked_benchmark       en    76 words
  paraphrased_benchmark  en    72 words
  harbour                en    79 words


(nb-prep-normalize)=
## 2. Unicode and Whitespace Normalisation

The corpus contains a document with a decomposed accent (`e` + U+0301), a zero-width space and a mojibake sequence. After NFC normalisation and clean-up, the composed and decomposed spellings are byte-identical, which is what deduplication and tokenization need.

In [2]:
_CONTROL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]")
_ZERO_WIDTH = re.compile(r"[​‌‍⁠﻿]")
_MULTI_SPACE = re.compile(r"[ \t ]+")
_MULTI_NEWLINE = re.compile(r"\n{3,}")


def normalize_text(text, form="NFC"):
    text = unicodedata.normalize(form, text)
    text = _CONTROL.sub("", text)
    text = _ZERO_WIDTH.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = _MULTI_SPACE.sub(" ", text)
    text = "\n".join(line.strip() for line in text.split("\n"))
    text = _MULTI_NEWLINE.sub("\n\n", text)
    return text.strip()


raw = dict((d, t) for d, t, _ in CORPUS)["mojibake"]
clean = normalize_text(raw)
print("raw code points:   ", len(raw))
print("clean code points: ", len(clean))
print("zero-width removed:", "​" in raw and "​" not in clean)
print("accents composed:  ", "é" in raw and "é" not in clean and "é" in clean)

assert "​" not in clean
assert "é" not in clean
# The mojibake "Ã©" is a *valid* string that NFC leaves alone; repairing it needs
# a tool such as ftfy that knows which byte sequences are likely decoding errors.
assert "Ã©" in clean
assert normalize_text("ﬁle Ａ", form="NFKC") == "file A"

raw code points:    504
clean code points:  500
zero-width removed: True
accents composed:   True


(nb-prep-langid)=
## 3. Language Identification with Character n-grams

A Cavnar-Trenkle profile classifier: rank the most frequent character 1- to 3-grams of each language's training text and score a document by how far its own ranking is from each profile. It is a toy next to fastText, but it uses the same features and fails in the same places, as the confidence-versus-length plot shows.

In [3]:
def char_ngrams(text, n_values=(1, 2, 3)):
    padded = f" {text.lower()} "
    counts = Counter()
    for n in n_values:
        for i in range(len(padded) - n + 1):
            counts[padded[i : i + n]] += 1
    return counts


def build_profile(texts, top_k=300):
    counts = Counter()
    for text in texts:
        counts.update(char_ngrams(text))
    return {gram: rank for rank, (gram, _) in enumerate(counts.most_common(top_k))}


def out_of_place_distance(doc_profile, lang_profile, top_k=300):
    return sum(abs(rank - lang_profile.get(gram, top_k)) for gram, rank in doc_profile.items())


def identify(text, profiles):
    doc = build_profile([text])
    scores = {lang: out_of_place_distance(doc, prof) for lang, prof in profiles.items()}
    ranked = sorted(scores.items(), key=lambda kv: kv[1])
    best, second = ranked[0], ranked[1]
    margin = (second[1] - best[1]) / max(1, second[1])  # 0 = tie, 1 = certain
    return best[0], margin


TRAIN = {
    "en": [
        "The quick brown fox jumps over the lazy dog while the farmer watches from the gate.",
        "Language models are trained on large amounts of text collected from the web.",
        "She walked to the station early in the morning and caught the first train north.",
        "Most of the houses on the street were built before the war and have small gardens.",
    ],
    "de": [
        "Der schnelle braune Fuchs springt über den faulen Hund, während der Bauer zusieht.",
        "Sprachmodelle werden mit großen Textmengen aus dem Internet trainiert.",
        "Sie ging früh am Morgen zum Bahnhof und nahm den ersten Zug nach Norden.",
        "Die meisten Häuser in der Straße wurden vor dem Krieg gebaut und haben kleine Gärten.",
    ],
}
PROFILES = {lang: build_profile(texts) for lang, texts in TRAIN.items()}

correct = 0
for doc_id, text, lang in CORPUS:
    pred, margin = identify(normalize_text(text), PROFILES)
    correct += pred == lang
    print(f"  {doc_id:<22} truth={lang} pred={pred} margin={margin:.2f}")
print(f"{correct}/{len(CORPUS)} correct")
assert identify(ARTICLE, PROFILES)[0] == "en"
assert identify(dict((d, t) for d, t, _ in CORPUS)["german"], PROFILES)[0] == "de"

  article                truth=en pred=en margin=0.17
  syndicated             truth=en pred=en margin=0.17
  retitled               truth=en pred=en margin=0.18
  mojibake               truth=en pred=en margin=0.16
  german                 truth=de pred=de margin=0.20
  menu                   truth=en pred=en margin=0.06
  spam                   truth=en pred=en margin=0.25
  boilerplate            truth=en pred=en margin=0.37
  catalogue              truth=en pred=en margin=0.00
  contact_list           truth=en pred=de margin=0.01
  prose_with_pii         truth=en pred=en margin=0.15
  leaked_benchmark       truth=en pred=en margin=0.20
  paraphrased_benchmark  truth=en pred=en margin=0.14
  harbour                truth=en pred=en margin=0.15
13/14 correct


In [4]:
# Confidence collapses as documents get shorter: the toy's margin on prefixes of the German document
german = dict((d, t) for d, t, _ in CORPUS)["german"]
lengths = [3, 5, 8, 12, 20, 30, 45, 60, 80]
margins = []
for n in lengths:
    prefix = " ".join(german.split()[:n])
    pred, margin = identify(prefix, PROFILES)
    margins.append(margin if pred == "de" else -margin)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(lengths, margins, marker="o")
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("document length (words)")
ax.set_ylabel("margin for the correct language")
ax.set_title("Language identification confidence vs. length")
plt.tight_layout()
plt.show()
assert margins[-1] > margins[0]

(nb-prep-filters)=
## 4. Quality and Repetition Filters

The Gopher document-level quality rules and the Gopher repetition rules, applied to the corpus. Each filter reports the first reason a document fails, which is what a pipeline records.

In [5]:
STOP_WORDS = {"the", "be", "to", "of", "and", "that", "have", "with"}
BULLETS = ("-", "*", "•", "·", "◦", "▪", "▫", "‣", "⁃")
_WORD = re.compile(r"\S+")
_ALPHA = re.compile(r"[A-Za-z]")


def gopher_quality_filter(text):
    words = _WORD.findall(text)
    n_words = len(words)
    if n_words < 50:
        return False, "too_few_words"
    if n_words > 100_000:
        return False, "too_many_words"
    mean_len = sum(len(w) for w in words) / n_words
    if not 3 <= mean_len <= 10:
        return False, "mean_word_length"
    hashes = text.count("#")
    ellipses = text.count("...") + text.count("…")
    if (hashes + ellipses) / n_words > 0.1:
        return False, "symbol_ratio"
    lines = [line for line in text.split("\n") if line.strip()]
    if lines:
        if sum(line.lstrip().startswith(BULLETS) for line in lines) / len(lines) > 0.9:
            return False, "bullet_lines"
        if sum(line.rstrip().endswith(("...", "…")) for line in lines) / len(lines) > 0.3:
            return False, "ellipsis_lines"
    if sum(bool(_ALPHA.search(w)) for w in words) / n_words < 0.8:
        return False, "non_alpha_words"
    lowered = {w.lower().strip(".,;:!?\"'()") for w in words}
    if len(STOP_WORDS & lowered) < 2:
        return False, "stop_words"
    return True, "ok"


def repetition_stats(text):
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    words = text.split()
    total_chars = max(1, sum(len(w) for w in words))
    stats = {}
    if lines:
        line_counts = Counter(lines)
        stats["dup_line_frac"] = sum(c - 1 for c in line_counts.values()) / len(lines)
        stats["dup_line_char_frac"] = sum(len(s) * (c - 1) for s, c in line_counts.items()) / max(
            1, sum(map(len, lines))
        )
    for n in (2, 3, 4):
        grams = Counter(tuple(words[i : i + n]) for i in range(len(words) - n + 1))
        if grams:
            gram, count = grams.most_common(1)[0]
            stats[f"top_{n}gram_char_frac"] = count * sum(map(len, gram)) / total_chars
    for n in range(5, 11):
        seen, dup_chars = set(), 0
        for i in range(len(words) - n + 1):
            gram = tuple(words[i : i + n])
            if gram in seen:
                dup_chars += sum(map(len, gram))
            seen.add(gram)
        stats[f"dup_{n}gram_char_frac"] = dup_chars / total_chars
    return stats


THRESHOLDS = {
    "dup_line_frac": 0.30,
    "dup_line_char_frac": 0.20,
    "top_2gram_char_frac": 0.20,
    "top_3gram_char_frac": 0.18,
    "top_4gram_char_frac": 0.16,
    **{f"dup_{n}gram_char_frac": t for n, t in zip(range(5, 11), (0.15, 0.14, 0.13, 0.12, 0.11, 0.10))},
}


def repetition_filter(text):
    stats = repetition_stats(text)
    for key, limit in THRESHOLDS.items():
        if stats.get(key, 0.0) > limit:
            return False, key
    return True, "ok"


print(f"{'document':<22} {'repetition':<22} quality")
results = {}
for doc_id, text, _ in CORPUS:
    rep = repetition_filter(text)
    qual = gopher_quality_filter(text)
    results[doc_id] = (rep, qual)
    print(f"{doc_id:<22} {rep[1]:<22} {qual[1]}")

assert results["article"] == ((True, "ok"), (True, "ok"))
assert results["harbour"] == ((True, "ok"), (True, "ok"))
assert not results["spam"][0][0]
assert not results["menu"][0][0] or not results["menu"][1][0]
assert results["catalogue"][1][1] in {"non_alpha_words", "stop_words", "mean_word_length"}

document               repetition             quality
article                ok                     ok
syndicated             ok                     ok
retitled               ok                     ok
mojibake               ok                     ok
german                 ok                     stop_words
menu                   dup_line_frac          bullet_lines
spam                   top_2gram_char_frac    stop_words
boilerplate            top_3gram_char_frac    ok
catalogue              ok                     non_alpha_words
contact_list           ok                     non_alpha_words
prose_with_pii         ok                     ok
leaked_benchmark       ok                     ok
paraphrased_benchmark  ok                     ok
harbour                ok                     ok


(nb-prep-perplexity)=
## 5. Perplexity Ranking

A smoothed unigram model trained on a few encyclopaedic sentences, standing in for CCNet's Wikipedia-trained KenLM. Ranking the corpus by perplexity puts formulaic text built from common words at the *head* (most predictable), prose in the middle, and catalogue noise at the tail, which is why the middle of the distribution is the part worth keeping. Spam whose vocabulary is absent from the reference also lands in the tail: the model cannot tell noise from an unfamiliar register.

In [6]:
class UnigramLM:
    def __init__(self, reference_texts, alpha=0.5):
        self.counts = Counter(w for t in reference_texts for w in re.findall(r"\w+", t.lower()))
        self.total = sum(self.counts.values())
        self.alpha = alpha
        self.vocab_size = len(self.counts) + 1

    def log_prob(self, word):
        c = self.counts.get(word.lower(), 0)
        return math.log((c + self.alpha) / (self.total + self.alpha * self.vocab_size))

    def perplexity(self, text):
        words = re.findall(r"\w+", text)
        if not words:
            return float("inf")
        return math.exp(-sum(self.log_prob(w) for w in words) / len(words))


REFERENCE = [
    "The river flows through the valley and into the sea near the old town.",
    "The city was founded in the twelfth century and grew around its port and market.",
    "Mathematics is the study of quantity, structure, space and change.",
    "The council published its report on the state of the local economy and the budget.",
    "A library is a collection of books and other materials made available to the public.",
    "The building was designed by a local architect and opened to residents in the spring.",
]
lm = UnigramLM(REFERENCE)
ranked = sorted(((lm.perplexity(t), d) for d, t, _ in CORPUS))
for ppl, doc_id in ranked:
    print(f"  {ppl:>10.1f}  {doc_id}")

order = [d for _, d in ranked]
# Formulaic text made of common words is the most predictable; prose sits in the middle;
# catalogue noise is at the tail. Spam built from words *absent* from the reference is
# also "noise" to this model, which is one more reason perplexity alone is not a filter.
assert order.index("boilerplate") < order.index("article") < order.index("catalogue")
assert order.index("article") < order.index("spam")

        28.6  boilerplate
        96.5  leaked_benchmark
       103.7  article
       103.7  syndicated
       105.6  mojibake
       107.4  retitled
       108.0  paraphrased_benchmark
       109.1  harbour
       133.8  prose_with_pii
       219.0  german
       224.0  catalogue
       224.0  contact_list
       224.0  menu
       224.0  spam


(nb-prep-dedup)=
## 6. Exact and Near-Duplicate Detection

Exact deduplication hashes normalised text. MinHash estimates Jaccard similarity over 5-gram shingles from a short signature, and LSH banding finds candidate pairs without comparing every pair. The probability that a pair with similarity $s$ becomes a candidate under $b$ bands of $r$ rows is

$$P(\text{candidate}) = 1 - (1 - s^r)^b$$

which is checked empirically below.

In [7]:
_MERSENNE = (1 << 61) - 1
_MAX_HASH = (1 << 32) - 1


def shingles(text, n=5):
    words = re.findall(r"\w+", text.lower())
    return {" ".join(words[i : i + n]) for i in range(max(0, len(words) - n + 1))}


def _hash32(s):
    return int.from_bytes(hashlib.sha1(s.encode("utf-8")).digest()[:4], "little")


class MinHash:
    def __init__(self, num_perm=128, seed=0):
        g = np.random.default_rng(seed)
        self.a = g.integers(1, _MERSENNE, size=num_perm, dtype=np.int64)
        self.b = g.integers(0, _MERSENNE, size=num_perm, dtype=np.int64)
        self.num_perm = num_perm

    def signature(self, shingle_set):
        if not shingle_set:
            return np.full(self.num_perm, _MAX_HASH, dtype=np.int64)
        x = np.array([_hash32(s) for s in shingle_set], dtype=np.int64)
        ax = (x[:, None].astype(object) * self.a.astype(object) + self.b.astype(object)) % _MERSENNE
        return (np.array(ax, dtype=np.int64) & _MAX_HASH).min(axis=0)

    @staticmethod
    def estimate_jaccard(sig_a, sig_b):
        return float(np.mean(sig_a == sig_b))


class LSHIndex:
    def __init__(self, num_perm=128, bands=16):
        assert num_perm % bands == 0
        self.bands, self.rows = bands, num_perm // bands
        self.buckets = [dict() for _ in range(bands)]

    def add(self, doc_id, sig):
        for band, table in enumerate(self.buckets):
            key = sig[band * self.rows : (band + 1) * self.rows].tobytes()
            table.setdefault(key, []).append(doc_id)

    def candidate_pairs(self):
        pairs = set()
        for table in self.buckets:
            for ids in table.values():
                for i in range(len(ids)):
                    for j in range(i + 1, len(ids)):
                        pairs.add((min(ids[i], ids[j]), max(ids[i], ids[j])))
        return pairs


def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0


texts = {d: normalize_text(t) for d, t, _ in CORPUS}

# Exact: the syndicated copy is caught, the retitled one is not
hashes = {d: hashlib.sha1(t.encode()).hexdigest() for d, t in texts.items()}
exact_dups = {d for d, h in hashes.items() if list(hashes.values()).index(h) != list(hashes).index(d)}
print("exact duplicates:", exact_dups)
assert exact_dups == {"syndicated"}

# MinHash + LSH: the retitled copy is caught too
mh = MinHash(num_perm=128)
sh = {d: shingles(t) for d, t in texts.items()}
sigs = {d: mh.signature(s) for d, s in sh.items()}
index = LSHIndex(num_perm=128, bands=16)
for d, s in sigs.items():
    index.add(d, s)
pairs = index.candidate_pairs()
print("LSH candidate pairs:")
for a, b in sorted(pairs):
    print(f"  {a:<12} {b:<12} true J={jaccard(sh[a], sh[b]):.2f}  est J={MinHash.estimate_jaccard(sigs[a], sigs[b]):.2f}")

assert ("article", "retitled") in pairs
assert ("article", "syndicated") in pairs
assert ("article", "harbour") not in pairs
assert abs(jaccard(sh["article"], sh["retitled"]) - MinHash.estimate_jaccard(sigs["article"], sigs["retitled"])) < 0.15

exact duplicates: {'syndicated'}


LSH candidate pairs:
  article      mojibake     true J=0.81  est J=0.77
  article      retitled     true J=0.92  est J=0.95
  article      syndicated   true J=1.00  est J=1.00
  mojibake     syndicated   true J=0.81  est J=0.77
  retitled     syndicated   true J=0.92  est J=0.95


In [8]:
# The LSH S-curve: theory vs. an empirical estimate from synthetic document pairs with controlled overlap
def synthetic_pair(similarity, n_shingles=400):
    # Two shingle sets whose Jaccard similarity is approximately `similarity`
    shared = int(n_shingles * 2 * similarity / (1 + similarity))
    only = n_shingles - shared
    base = [f"s{i}" for i in range(shared)]
    a = set(base) | {f"a{i}" for i in range(only)}
    b = set(base) | {f"b{i}" for i in range(only)}
    return a, b


def candidate_probability(similarity, bands, rows):
    return 1 - (1 - similarity**rows) ** bands


sims = np.linspace(0.05, 0.95, 19)
settings = [(8, 16), (16, 8), (32, 4)]
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for bands, rows in settings:
    theory = [candidate_probability(s, bands, rows) for s in sims]
    ax.plot(sims, theory, label=f"theory b={bands}, r={rows}")
empirical = []
for s in sims:
    hits = 0
    trials = 12
    for trial in range(trials):
        a, b = synthetic_pair(s)
        m = MinHash(num_perm=128, seed=trial)
        idx = LSHIndex(num_perm=128, bands=16)
        idx.add("a", m.signature(a))
        idx.add("b", m.signature(b))
        hits += ("a", "b") in idx.candidate_pairs()
    empirical.append(hits / trials)
ax.plot(sims, empirical, "ko", label="empirical b=16, r=8")
ax.set_xlabel("Jaccard similarity")
ax.set_ylabel("P(candidate pair)")
ax.set_title("LSH banding: probability of becoming a candidate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# The empirical curve for b=16, r=8 should sit on its theory: low at 0.3, high at 0.9
assert empirical[sims.tolist().index(min(sims, key=lambda v: abs(v - 0.3)))] < 0.3
assert empirical[-1] > 0.9

In [9]:
# Substring deduplication: a shared licence header found with a (naive) suffix array
def repeated_spans(tokens, min_len=50):
    n = len(tokens)
    suffix_array = sorted(range(n), key=lambda i: tokens[i:])
    spans = []
    for k in range(1, n):
        i, j = suffix_array[k - 1], suffix_array[k]
        lcp = 0
        while i + lcp < n and j + lcp < n and tokens[i + lcp] == tokens[j + lcp]:
            lcp += 1
        if lcp >= min_len:
            spans.append((j, j + lcp))
    return sorted(set(spans))


licence = (
    "permission is hereby granted free of charge to any person obtaining a copy of this "
    "software and associated documentation files to deal in the software without restriction"
).split()
file_a = "module alpha".split() + licence + "def main(): pass".split()
file_b = "module beta".split() + licence + "class Thing: pass".split()
tokens = file_a + file_b
spans = repeated_spans(tokens, min_len=len(licence))
print("repeated spans:", spans)
assert len(spans) == 1 and tokens[spans[0][0] : spans[0][1]] == licence

repeated spans: [(2, 29)]


(nb-prep-pii)=
## 7. Personal-Data Redaction

Regular expressions for email addresses, North American phone numbers and IPv4 addresses, with a document-level rule that drops anything holding more than five matches. The second cell shows the patterns both over- and under-matching on international formats and version strings.

In [10]:
PII_PATTERNS = {
    "EMAIL": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"),
    "PHONE": re.compile(r"(?<!\w)(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{3}\)?[\s.-]?)\d{3}[\s.-]?\d{4}(?!\w)"),
    "IPV4": re.compile(r"(?<!\d)(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)(?!\d)"),
}


def redact_pii(text, max_matches=5):
    counts = {}
    for label, pattern in PII_PATTERNS.items():
        text, n = pattern.subn(f"|||{label}|||", text)
        counts[label] = n
    return text, counts, sum(counts.values()) <= max_matches


for doc_id in ("prose_with_pii", "contact_list"):
    redacted, counts, keep = redact_pii(texts[doc_id])
    print(f"{doc_id}: {counts} keep={keep}")
    print("   ", redacted[:120].replace("\n", " | "), "...")

r, c, keep = redact_pii(texts["prose_with_pii"])
assert c == {"EMAIL": 1, "PHONE": 1, "IPV4": 1} and keep
assert "example.org" not in r and "10.0.0.12" not in r
assert redact_pii(texts["contact_list"])[2] is False

prose_with_pii: {'EMAIL': 1, 'PHONE': 1, 'IPV4': 1} keep=True
    For questions about the schedule, email the coordinator at |||EMAIL||| or call |||PHONE||| during office hours. The serv ...
contact_list: {'EMAIL': 10, 'PHONE': 10, 'IPV4': 0} keep=False
    Member 0: |||EMAIL|||, |||PHONE||| | Member 1: |||EMAIL|||, |||PHONE||| | Member 2: |||EMAIL|||, |||PHONE||| | Member 3: |||EM ...


In [11]:
tricky = {
    "uk phone": "Call +44 20 7946 0958 for details.",
    "german phone": "Rufen Sie 030 123456 an.",
    "version string": "Upgrade to release 2.14.0.1 before Friday.",
    "iso timestamp": "Logged at 2024-05-01 at 12.30.45 server time.",
    "isbn": "ISBN 978-0-306-40615-7 is the paperback.",
}
for name, text in tricky.items():
    _, counts, _ = redact_pii(text)
    found = {k: v for k, v in counts.items() if v}
    print(f"  {name:<16} {found or 'nothing matched'}")

# The honest result: a UK number in international format is caught only partially or
# not at all, a German local number is missed, and a dotted version string can be
# mistaken for an IPv4 address. Patterns need per-locale tuning and precision checks.
assert redact_pii(tricky["german phone"])[1]["PHONE"] == 0

  uk phone         nothing matched
  german phone     nothing matched
  version string   {'IPV4': 1}
  iso timestamp    nothing matched
  isbn             nothing matched


(nb-prep-decontam)=
## 8. Benchmark Decontamination

A 13-gram index built from a toy benchmark finds the document that quotes a question verbatim and misses the paraphrase, which is the known limit of $n$-gram decontamination.

In [12]:
def ngram_set(text, n):
    words = re.findall(r"\w+", text.lower())
    return {tuple(words[i : i + n]) for i in range(max(0, len(words) - n + 1))}


def contamination_index(examples, n=13):
    index = set()
    for ex in examples:
        grams = ngram_set(ex, n) or {tuple(re.findall(r"\w+", ex.lower()))}
        index |= grams
    return index


def contaminated_fraction(doc, index, n=13):
    grams = ngram_set(doc, n)
    return len(grams & index) / len(grams) if grams else 0.0


BENCHMARK = [
    "What is the capital of the country whose largest city is Istanbul and whose currency is the lira?",
    "Which element has atomic number 26?",
]
idx13 = contamination_index(BENCHMARK, n=13)
idx8 = contamination_index(BENCHMARK, n=8)
print(f"{'document':<22} {'13-gram':>8} {'8-gram':>8}")
for d, t in texts.items():
    f13, f8 = contaminated_fraction(t, idx13, 13), contaminated_fraction(t, idx8, 8)
    if f13 or f8:
        print(f"{d:<22} {f13:>8.3f} {f8:>8.3f}")

assert contaminated_fraction(texts["leaked_benchmark"], idx13, 13) > 0
assert contaminated_fraction(texts["paraphrased_benchmark"], idx13, 13) == 0.0
assert contaminated_fraction(texts["paraphrased_benchmark"], idx8, 8) == 0.0  # still missed at n=8
assert contaminated_fraction(texts["harbour"], idx8, 8) == 0.0

document                13-gram   8-gram
leaked_benchmark          0.094    0.159


(nb-prep-pipeline)=
## 9. The Complete Pipeline

All stages in the order a production pipeline runs them, with per-stage drop counts. Deduplication is the one stage that needs state across documents; here it is a closure over an exact-hash set and an LSH index, which is what a distributed implementation does with a shuffle.

In [13]:
class Pipeline:
    def __init__(self, stages):
        self.stages = stages
        self.counts = Counter()
        self.reasons = Counter()

    def run(self, docs):
        for doc in docs:
            self.counts["input"] += 1
            for name, fn in self.stages:
                doc, reason = fn(doc)
                if doc is None:
                    self.counts[f"dropped_by:{name}"] += 1
                    self.reasons[f"{name}:{reason}"] += 1
                    break
            else:
                self.counts["output"] += 1
                yield doc

    def report(self):
        lines = [f"input   {self.counts['input']}"]
        for name, _ in self.stages:
            lines.append(f"  -{self.counts[f'dropped_by:{name}']:<4} {name}")
        lines.append(f"output  {self.counts['output']}")
        return "\n".join(lines)


def stage_normalize(doc):
    return {**doc, "text": normalize_text(doc["text"])}, "ok"


def stage_language(doc, target="en"):
    lang, margin = identify(doc["text"], PROFILES)
    return ({**doc, "lang": lang}, "ok") if lang == target else (None, f"lang={lang}")


def stage_repetition(doc):
    ok, reason = repetition_filter(doc["text"])
    return (doc, "ok") if ok else (None, reason)


def stage_quality(doc):
    ok, reason = gopher_quality_filter(doc["text"])
    return (doc, "ok") if ok else (None, reason)


def make_dedup_stage(num_perm=128, bands=16):
    seen_hashes, mh, index = set(), MinHash(num_perm), LSHIndex(num_perm, bands)

    def stage(doc):
        digest = hashlib.sha1(doc["text"].encode()).hexdigest()
        if digest in seen_hashes:
            return None, "exact"
        seen_hashes.add(digest)
        sig = mh.signature(shingles(doc["text"]))
        before = index.candidate_pairs()
        index.add(doc["id"], sig)
        if index.candidate_pairs() - before:
            return None, "near"
        return doc, "ok"

    return stage


def stage_pii(doc):
    text, counts, keep = redact_pii(doc["text"])
    return ({**doc, "text": text, "pii": counts}, "ok") if keep else (None, "pii_dense")


def stage_decontaminate(doc, index=contamination_index(BENCHMARK, 13)):
    return (doc, "ok") if contaminated_fraction(doc["text"], index, 13) == 0 else (None, "benchmark_overlap")


pipe = Pipeline(
    [
        ("normalize", stage_normalize),
        ("language", stage_language),
        ("repetition", stage_repetition),
        ("quality", stage_quality),
        ("dedup", make_dedup_stage()),
        ("pii", stage_pii),
        ("decontaminate", stage_decontaminate),
    ]
)
kept = list(pipe.run({"id": d, "text": t} for d, t, _ in CORPUS))
print(pipe.report())
print()
print("drop reasons:")
for key, n in sorted(pipe.reasons.items()):
    print(f"  {n}  {key}")
print()
print("kept:", [d["id"] for d in kept])

kept_ids = {d["id"] for d in kept}
assert "article" in kept_ids and "harbour" in kept_ids and "prose_with_pii" in kept_ids
assert "paraphrased_benchmark" in kept_ids  # the known miss
assert kept_ids.isdisjoint({"syndicated", "retitled", "german", "menu", "spam", "boilerplate", "catalogue", "contact_list", "leaked_benchmark"})
assert pipe.counts["input"] == len(CORPUS)
assert pipe.counts["output"] == len(kept)
assert pipe.reasons["dedup:exact"] == 1 and pipe.reasons["dedup:near"] >= 1
assert pipe.reasons["decontaminate:benchmark_overlap"] == 1

input   14
  -0    normalize
  -2    language
  -3    repetition
  -1    quality
  -3    dedup
  -0    pii
  -1    decontaminate
output  4

drop reasons:
  1  decontaminate:benchmark_overlap
  1  dedup:exact
  2  dedup:near
  2  language:lang=de
  1  quality:non_alpha_words
  1  repetition:dup_line_frac
  1  repetition:top_2gram_char_frac
  1  repetition:top_3gram_char_frac

kept: ['article', 'prose_with_pii', 'paraphrased_benchmark', 'harbour']


## Summary

- Normalisation makes equivalent strings byte-identical, which every later stage relies on; it does not repair mojibake, which needs a dedicated tool.
- Character n-gram language identification is accurate on paragraphs and unreliable on a handful of words.
- The Gopher quality and repetition rules remove menus, spam and catalogues from the corpus with no learned component, and report *why*.
- Perplexity under a reference model ranks boilerplate first, prose in the middle and noise last, so it is a bucketing tool rather than a hard filter.
- Exact hashing catches verbatim copies; MinHash with LSH catches lightly edited ones at a cost of one short signature per document, with a candidate probability that follows $1 - (1 - s^r)^b$.
- Regular-expression redaction is precise on well-formatted values and weak elsewhere; dense-PII documents are dropped outright.
- 13-gram decontamination catches verbatim benchmark leaks and misses paraphrases.
- The assembled pipeline keeps 4 of 14 documents and accounts for every one it dropped.